# 05 — Bayesian Retention A/B Test

**Business question:** Does the retention offer reduce churn, and is the evidence strong enough to deploy it?

The included experiment is **simulated** because IBM Telco does not include randomized treatment data.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.decision import bayesian_ab_churn, posterior_campaign_profit

path = ROOT/"data/processed/demo_retention_experiment.csv"
if not path.exists():
    exec((ROOT/"data/generate_demo_data.py").read_text())
    main()
exp = pd.read_csv(path)
exp.groupby("treatment")["churn_after_campaign"].agg(["count","sum","mean"])

In [ ]:
control = exp[exp.treatment==0]["churn_after_campaign"]
treat = exp[exp.treatment==1]["churn_after_campaign"]

result = bayesian_ab_churn(
    churn_control=int(control.sum()),
    n_control=len(control),
    churn_treat=int(treat.sum()),
    n_treat=len(treat),
)
{k:v for k,v in result.items() if "samples" not in k}

In [ ]:
plt.hist(result["reduction_samples"], bins=60, density=True)
plt.axvline(0)
plt.xlabel("Control churn - Treatment churn")
plt.title("Posterior distribution of absolute churn reduction")
plt.tight_layout()
plt.show()

In [ ]:
treated = exp[exp.treatment==1]
profit = posterior_campaign_profit(
    result["reduction_samples"],
    n_targeted=len(treated),
    monthly_margin=treated["expected_monthly_margin"].mean(),
    retained_months=treated["expected_remaining_months"].mean(),
    cost_per_customer=30,
)
{k:v for k,v in profit.items() if v is not profit["profit_samples"]}

## Decision framing

Report:
- posterior probability treatment reduces churn,
- expected absolute reduction,
- 95% credible interval,
- expected campaign profit,
- probability campaign is profitable.

This is more decision-relevant than reporting only a p-value.